[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/terrell/terrell.ipynb)

# Terrell Rotation

This example renders relativistic photographs of a sphere and a cube passing a stationary camera at nearly the speed of light. Although Lorentz contraction shortens a moving body along its direction of motion on an equal-time slice, a camera records photons that arrive simultaneously rather than photons that left the body simultaneously. Light from the far and trailing sides of the body sets out earlier than light from the near side, when the body lay further upstream, and that travel-time delay combines with Lorentz contraction to rotate the apparent view of the body while keeping the outline of a sphere circular.

In spacetime algebra, events and light rays are represented as intrinsic spacetime directions, with arriving photons satisfying `ray.squared() == 0`. Transforming an emission event into the body's rest frame via `frame << centre` locates where the camera sits beside the stationary body, where photons travel along straight spatial offsets, and boosting those rays back via `frame >> rays` produces the camera's photograph. Because a Lorentz boost acts linearly on spacetime directions, rays lying in a hyperplane `plane & ray == 0` remain in a hyperplane in every inertial frame, mapping circles on the camera's sky to circles.

In three-vector special-relativity notation ([Hornof et al., 2025](https://arxiv.org/abs/2409.04296)), transforming a photon arrival angle $\theta$ from the camera frame to $\theta'$ in the body rest frame applies the relativistic aberration formula $\cos\theta' = \frac{\cos\theta + \beta}{1 + \beta\cos\theta}$ together with explicit retarded-time roots $t_{\mathrm{ret}}$ for each surface point. Viewing the body perpendicular to its motion at $\theta = \pi/2$ evaluates to $\cos\theta' = \beta$, matching a rest-frame photograph taken from an angle rotated backward by $\arcsin\beta$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga

from examples.relativity.terrell import render
from examples.util.animation import show_states

np.set_printoptions(precision=1, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Antivector = ga.gatype.antivector()                 # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Rotor = ga.gatype.rotor()                           # Rotor+

time = -mv.t.dual()                                 # [] Antivector-
space = stack([mv.x, mv.y, mv.z]).dual()            # [3] Antivector-
travel, line_of_sight, up = space

## Camera position in the body frame

A camera at the origin photographs the passing body when light from the body's centre arrives along a chosen sighting angle. Tracing that arriving ray backward along the past light cone to the body's track locates the emission event, and transforming that event into the body's rest frame places the camera behind the perpendicular by a fraction equal to the body's speed.

In [ ]:
track_distance = 4.5
body_speeds = np.array([0.0, 0.6, 0.9])

In [ ]:
def boost_rotors(speeds: np.ndarray) -> Rotor:
    """Hyperbolic turns in the travel-time plane map the body's rest frame to moving frames (`[...] Rotor+`)."""
    return ((travel & time).dual() * (np.arctanh(speeds) / 2)).exp()            # [...] Rotor+


def sighting_events(distance: float, angles: np.ndarray) -> Antivector:
    """Past-light-cone rays back from the camera reach the track at `distance` along each sighting angle (`[...] Antivector-`)."""
    turn = (mv.xy * (-angles / 2)).exp()                                        # [...] Rotor+
    sight = turn >> travel                                                      # [...] Antivector-
    return (sight - time) * (distance / (mv.y & sight))                         # [...] Antivector-

In [ ]:
def rest_camera(frame: Rotor, centre: Antivector) -> Antivector:
    """Boosting the emission event `centre` into the body's frame gives the camera's spatial offset (`[...] Antivector-`)."""
    camera = -(frame << centre)                                                 # [...] Antivector-
    return camera - (mv.t & camera) * time                                      # [...] Antivector-


frames = boost_rotors(body_speeds)                                              # [3] Rotor+
sideways_camera = rest_camera(frames, sighting_events(track_distance, np.pi / 2))  # [3] Antivector-
leaning = (mv.x & sideways_camera) / sideways_camera.antinorm()                 # [3] Scalar+
print("apparent turn angles (deg):", np.degrees(np.arcsin(-leaning.to_array())))

## Photographs of a passing sphere and cube

Stereographic projection of the camera's sky maps circles on the sky to circles on the image plane. Each pixel ray transforms into the body's rest frame via `frame << ray`, where it runs straight from the stationary camera offset to the first surface it meets on the sphere or cube.

In [ ]:
sphere_radius = 1.0
half_width = 1.0
height = 1.6
meridian_spacing = np.pi / 4
parallel_spacing = np.pi / 6
parallel_reach = np.pi / 3
n_frames = 108
sightings = np.linspace(3 * np.pi / 4, np.pi / 4, n_frames)
frame_duration_ms = 40

In [ ]:
def camera_positions(frame: Rotor, distance: float, angles: np.ndarray) -> Iterator[Antivector]:
    """Yields the camera's spatial position in each body frame (`[...] Antivector-`) across the sequence of sighting `angles`."""
    for angle in angles:
        yield rest_camera(frame, sighting_events(distance, angle))


sphere = height * up                                                            # [] Antivector-
cube = -height * up                                                             # [] Antivector-
camera_states = show_states(
    camera_positions(frames, track_distance, sightings),
    render.photographs_animation,
    frames, sphere, sphere_radius, cube, half_width, meridian_spacing, parallel_spacing, parallel_reach,
    name="terrell_rotation", duration_ms=frame_duration_ms,
)

From top to bottom, the three lanes show the passing pair at rest, at six tenths of the speed of light, and at nine tenths of the speed of light whenever the pair's centre crosses the same sighting angle. Red colours the leading cube face, blue colours the trailing face, and violet colours the face turned toward the trackside camera. At rest the cube shows its leading red face on approach, its violet side at closest sight, and its trailing blue face as it recedes, whereas at nine tenths of the speed of light the trailing blue face remains visible almost throughout the pass while the sphere's meridian grid rotates and its silhouette stays circular.

## Circular sphere outlines on the sky

In the body's rest frame, rays from the camera graze the sphere along a spatial circle orthogonal to the separation line. Boosting those null rays to the camera's frame and normalizing each to unit arrival time keeps every quadruple of outline rays coplanar, so their four-way regressive join vanishes at every speed.

In [ ]:
n_outline = 96
outline_angles = np.linspace(0, 2 * np.pi, n_outline, endpoint=False)

In [ ]:
def rest_rays(camera: Antivector, points: Antivector) -> Antivector:
    """Null rays in the body's rest frame run straight from `points` to `camera` (`[...] Antivector-`)."""
    offsets = camera - points                                                   # [...] Antivector-
    return offsets + time * offsets.antinorm()                                  # [...] Antivector-


def grazing_outline(camera: Antivector, centre: Antivector, radius: float, angles: np.ndarray) -> Antivector:
    """Rays from `camera` graze a sphere around `centre` along a spatial circle (`[...] Antivector-`)."""
    offset = camera - centre                                                    # [...] Antivector-
    distance = offset.antinorm()                                                # [...] Scalar+
    # Build the grazing ring around `-line_of_sight` and rotate it onto the direction from the sphere to the camera.
    alignment = ((offset / distance) * (-line_of_sight).inverse()).square_root()  # [...] Rotor+
    nearest = (radius / distance) * -line_of_sight + (1 - (radius / distance).squared()).square_root() * up
    ring_rotors = (mv.zx * (angles / 2)).exp()                                  # [...] Rotor+
    ring = ring_rotors >> nearest[..., None]                                    # [...] Antivector-
    return centre + radius * (alignment[..., None] >> ring)                     # [...] Antivector-


first_camera = camera_states[0]                                                 # [3] Antivector-
grazing_points = grazing_outline(first_camera, sphere, sphere_radius, outline_angles)  # [3, n_outline] Antivector-
boosted_rays = frames[:, None] >> rest_rays(first_camera[:, None], grazing_points)     # [3, n_outline] Antivector-
sky_points = boosted_rays / (mv.t & boosted_rays)                               # [3, n_outline] Antivector-
four_ray_join = sky_points[:, 0] & sky_points[:, 20] & sky_points[:, 50] & sky_points[:, 70]  # [3] Scalar+

In [ ]:
# checks
assert ((leaning + body_speeds).abs() < 1e-8).all()
assert (four_ray_join.abs() < 1e-8).all()